In [ ]:
import pandas as pd
import geopandas as gpd
import os
from pathlib import Path

def process_csv_to_shapefiles_with_geometry_matching(predictions_base_path, segmentation_base_path):
    """
    Process CSV files from 2019-2023 and convert to shapefiles by predicted_layer
    Uses segmentation shapefiles for geometry by matching PXLVAL
    
    Parameters:
    predictions_base_path (str): Base path containing the prediction folders
    segmentation_base_path (str): Base path containing the segmentation shapefiles
    """
    
    # Years to process
    years = range(2019, 2025)  # 2019 to 2024
    
    # Base paths
    predictions_dir = Path(predictions_base_path)
    segmentation_dir = Path(segmentation_base_path)
    
    # Create output directory
    output_dir = predictions_dir / "masked_Shapefile_Predicted"
    output_dir.mkdir(exist_ok=True)
    
    # Process each year
    for year in years:
        print(f"Processing year {year}...")
        
        # Prediction CSV path
        predictions_folder = f"Predictions_{year}_Enhanced_Seasonal"
        csv_path = predictions_dir / predictions_folder / f"predictions_{year}.csv"
        
        # Segmentation shapefile path
        seg_folder = f"Seg_{year}"
        shapefile_path = segmentation_dir / seg_folder / "segmentation_layer.shp"
        
        # Check if both files exist
        if not csv_path.exists():
            print(f"Warning: Predictions CSV {csv_path} does not exist. Skipping {year}.")
            continue
            
        if not shapefile_path.exists():
            print(f"Warning: Segmentation shapefile {shapefile_path} does not exist. Skipping {year}.")
            continue
        
        try:
            # Read CSV file with low_memory=False to avoid dtype warnings
            print(f"  Reading CSV: {csv_path}")
            df_predictions = pd.read_csv(csv_path, low_memory=False)
            
            # Check if required columns exist
            required_cols = ['PXLVAL', 'predicted_layer', 'prediction_confidence']
            missing_cols = [col for col in required_cols if col not in df_predictions.columns]
            
            if missing_cols:
                print(f"Warning: Missing columns in {year}: {missing_cols}. Skipping {year}.")
                continue
            
            # Select only required columns
            df_predictions = df_predictions[required_cols].copy()
            
            # Remove duplicates based on PXLVAL (keep first occurrence)
            df_predictions = df_predictions.drop_duplicates(subset=['PXLVAL'], keep='first')
            
            # Read segmentation shapefile
            print(f"  Reading shapefile: {shapefile_path}")
            gdf_segmentation = gpd.read_file(shapefile_path)
            
            # Check if PXLVAL exists in segmentation shapefile
            if 'PXLVAL' not in gdf_segmentation.columns:
                print(f"Warning: PXLVAL column not found in segmentation shapefile for {year}. Available columns: {list(gdf_segmentation.columns)}")
                continue
            
            # Merge predictions with segmentation geometry based on PXLVAL
            print(f"  Merging data based on PXLVAL...")
            gdf_merged = gdf_segmentation[['PXLVAL', 'geometry']].merge(
                df_predictions, 
                on='PXLVAL', 
                how='inner'
            )
            
            print(f"  Matched {len(gdf_merged)} records out of {len(df_predictions)} predictions")
            
            if len(gdf_merged) == 0:
                print(f"Warning: No matching PXLVAL found between CSV and shapefile for {year}. Skipping.")
                continue
            
            # Rename columns to be shapefile-friendly (10 character limit)
            column_mapping = {
                'PXLVAL': 'PXLVAL',  # Already 6 chars, OK
                'predicted_layer': 'PRED_LAYER',  # Shortened to 10 chars
                'prediction_confidence': 'PRED_CONF'  # Shortened to 9 chars
            }
            
            gdf_merged = gdf_merged.rename(columns=column_mapping)
            
            # Get unique predicted layers
            unique_layers = gdf_merged['PRED_LAYER'].unique()
            
            print(f"  Found {len(unique_layers)} unique predicted layers: {unique_layers}")
            
            # Create separate shapefile for each predicted layer
            for layer in unique_layers:
                # Filter data for current layer
                layer_gdf = gdf_merged[gdf_merged['PRED_LAYER'] == layer].copy()
                
                # Create filename (replace any problematic characters)
                safe_layer_name = str(layer).replace(' ', '_').replace('/', '_').replace('\\', '_').replace('.', '_')
                shapefile_name = f"{year}_{safe_layer_name}"
                shapefile_path = output_dir / f"{shapefile_name}.shp"
                
                # Save as shapefile
                try:
                    layer_gdf.to_file(shapefile_path, driver='ESRI Shapefile')
                    print(f"  Created: {shapefile_name}.shp ({len(layer_gdf)} records)")
                except Exception as e:
                    print(f"  Error creating shapefile for {year}, layer {layer}: {e}")
            
        except Exception as e:
            print(f"Error processing {year}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    print(f"\nProcessing complete! Shapefiles saved in: {output_dir}")
    
    # Print summary
    shapefiles = list(output_dir.glob("*.shp"))
    print(f"Total shapefiles created: {len(shapefiles)}")
    
    # List all created shapefiles
    if shapefiles:
        print("\nCreated shapefiles:")
        for shp in sorted(shapefiles):
            print(f"  {shp.name}")

def inspect_data_structure(predictions_base_path, segmentation_base_path, year=2019):
    """
    Helper function to inspect the data structure for debugging
    """
    print(f"Inspecting data structure for year {year}...")
    
    predictions_dir = Path(predictions_base_path)
    segmentation_dir = Path(segmentation_base_path)
    
    # Check prediction CSV
    predictions_folder = f"Predictions_{year}_Enhanced_Seasonal"
    csv_path = predictions_dir / predictions_folder / f"predictions_{year}.csv"
    
    if csv_path.exists():
        print(f"\nPrediction CSV: {csv_path}")
        df = pd.read_csv(csv_path, nrows=5, low_memory=False)
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print(f"First few PXLVAL values: {df['PXLVAL'].head().tolist()}")
        print(f"Unique predicted_layer values (first 10): {df['predicted_layer'].unique()[:10]}")
        
        if 'geometry' in df.columns:
            print(f"Sample geometry values:")
            for i, geom in enumerate(df['geometry'].head(3)):
                print(f"  {i+1}: {str(geom)[:100]}...")
    
    # Check segmentation shapefile
    seg_folder = f"Seg_{year}"
    shapefile_path = segmentation_dir / seg_folder / "segmentation_layer.shp"
    
    if shapefile_path.exists():
        print(f"\nSegmentation Shapefile: {shapefile_path}")
        gdf = gpd.read_file(shapefile_path)
        print(f"Shape: {gdf.shape}")
        print(f"Columns: {list(gdf.columns)}")
        print(f"CRS: {gdf.crs}")
        if 'PXLVAL' in gdf.columns:
            print(f"First few PXLVAL values: {gdf['PXLVAL'].head().tolist()}")
            print(f"PXLVAL data type: {gdf['PXLVAL'].dtype}")
        print(f"Geometry type: {gdf.geometry.geom_type.unique()}")

def main():
    # Set your paths here
    predictions_base_path = r"F:\_____My_Thesies____\Implimantation\3.Train_Model\4.AugModel_On_No_F-E_Data\__Predict"
    segmentation_base_path = r"F:\_____My_Thesies____\_ImpFiles\3. Feature Selection\_Sentinel Data\2. Divided Data\6.Segmentation"
    
    # Uncomment the next line to inspect data structure first
    # inspect_data_structure(predictions_base_path, segmentation_base_path, 2019)
    
    # Process the files
    process_csv_to_shapefiles_with_geometry_matching(predictions_base_path, segmentation_base_path)

if __name__ == "__main__":
    main()

Processing year 2024...
  Reading CSV: F:\_____My_Thesies____\Implimantation\3.Train_Model\4.AugModel_On_No_F-E_Data\__Predict\Predictions_2024_Enhanced_Seasonal\predictions_2024.csv
  Reading shapefile: F:\_____My_Thesies____\_ImpFiles\3. Feature Selection\_Sentinel Data\2. Divided Data\6.Segmentation\Seg_2024\segmentation_layer.shp
  Merging data based on PXLVAL...
  Matched 6311 records out of 6312 predictions
  Found 13 unique predicted layers: ['Phragmites-polygon' 'Wheat-polygon' 'Alfalfa-polygon'
 'Degreed_Wetland_Poly-polygon' 'Tamarix-polygon' 'Urban-polygon'
 'Clover-polygon' 'Mixed_TP-polygon' 'Rice-polygon' 'Corn-polygon'
 'Bare-polygon' 'Plowed-polygon' 'Sugar_Beet-polygon']
  Created: 2024_Phragmites-polygon.shp (188 records)
  Created: 2024_Wheat-polygon.shp (3634 records)
  Created: 2024_Alfalfa-polygon.shp (778 records)
  Created: 2024_Degreed_Wetland_Poly-polygon.shp (105 records)
  Created: 2024_Tamarix-polygon.shp (62 records)
  Created: 2024_Urban-polygon.shp (189 

### Create Zip file

In [7]:
import os
import zipfile
from pathlib import Path

def create_shapefile_zips(directory_path):
    """
    Create zip files for each shapefile in the specified directory.
    Each shapefile (with all its associated files) will be zipped individually.
    """
    # Convert to Path object for easier handling
    base_dir = Path(directory_path)
    
    # Check if directory exists
    if not base_dir.exists():
        print(f"Directory {directory_path} does not exist!")
        return
    
    # Get all .shp files
    shp_files = list(base_dir.glob("*.shp"))
    
    if not shp_files:
        print("No shapefiles found in the directory!")
        return
    
    print(f"Found {len(shp_files)} shapefiles to process...")
    
    for shp_file in shp_files:
        # Get the base name without extension
        base_name = shp_file.stem
        
        # Create zip file name
        zip_filename = base_dir / f"{base_name}.zip"
        
        # Get all files with the same base name (different extensions)
        related_files = list(base_dir.glob(f"{base_name}.*"))
        
        # Create zip file
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for file in related_files:
                # Only include files with common shapefile extensions
                if file.suffix.lower() in ['.shp', '.shx', '.dbf', '.prj', '.sbn', '.sbx', '.fbn', '.fbx', '.ain', '.aih', '.ixs', '.mxs', '.atx', '.shp.xml', '.cpg', '.qix']:
                    # Add file to zip with just the filename (not full path)
                    zipf.write(file, file.name)
                    print(f"  Added {file.name} to {zip_filename.name}")
        
        print(f"Created zip file: {zip_filename}")
    
    print("All shapefiles have been zipped successfully!")

# Usage
directory_path = r"F:\_____My_Thesies____\Implimantation\3.Train_Model\4.AugModel_On_No_F-E_Data\__Predict\masked_Shapefile_Predicted"
create_shapefile_zips(directory_path)

Found 78 shapefiles to process...
  Added 2019_Alfalfa-polygon.cpg to 2019_Alfalfa-polygon.zip
  Added 2019_Alfalfa-polygon.dbf to 2019_Alfalfa-polygon.zip
  Added 2019_Alfalfa-polygon.prj to 2019_Alfalfa-polygon.zip
  Added 2019_Alfalfa-polygon.shp to 2019_Alfalfa-polygon.zip
  Added 2019_Alfalfa-polygon.shx to 2019_Alfalfa-polygon.zip
Created zip file: F:\_____My_Thesies____\Implimantation\3.Train_Model\4.AugModel_On_No_F-E_Data\__Predict\masked_Shapefile_Predicted\2019_Alfalfa-polygon.zip
  Added 2019_Bare-polygon.cpg to 2019_Bare-polygon.zip
  Added 2019_Bare-polygon.dbf to 2019_Bare-polygon.zip
  Added 2019_Bare-polygon.prj to 2019_Bare-polygon.zip
  Added 2019_Bare-polygon.shp to 2019_Bare-polygon.zip
  Added 2019_Bare-polygon.shx to 2019_Bare-polygon.zip
Created zip file: F:\_____My_Thesies____\Implimantation\3.Train_Model\4.AugModel_On_No_F-E_Data\__Predict\masked_Shapefile_Predicted\2019_Bare-polygon.zip
  Added 2019_Clover-polygon.cpg to 2019_Clover-polygon.zip
  Added 2019_C